This notebook demonstrates how the human ME Model biomass objective is formulated from the Recon2.2 objective

In [1]:
import cobra

In [28]:
# load recon2.2
build_files_path = '/data2/hratch/human_me/build/'
full_model = cobra.io.read_sbml_model('/data2/hratch/human_me/prebuild/recon2_2.xml')

# 1: Metabolic Model (Recon2.2) Implementation

## 1.0 Metabolic Model Units

Some basics of units in metabolic models:

1) Stoichiometric coefficients and fluxes carry the units in a metabolic model, since they represent actual values used by the algorithm. 

2) There is a distinction between the flux units (which are always hr<sup>-1</sup>) and the metabolite consumption/production rates, which are in [*flux units*]x[*stoichiometric coefficient units*]. 

3) Stoichiometric coefficients represent the units of the metabolite object (as a metabolite concentration). For a metabolite X, units are typically in $$\frac{\text{mmol X}}{\text{1g DW}_{\text{cell}}}$$ Then, a metabolite product/consumption rate is typicially $$\frac{\text{mmol X}}{\text{1g DW}_{\text{cell}}*\text{hr}}$$

4) The molecular weight of metabolite X is in $$\text{kDA} = \frac{g}{mmol}$$ s.t. metabolite concentration can be converted to $$\frac{\text{g X}}{\text{1g DW}_{\text{cell}}}$$ by multiplying the stoichiometric coefficient of X by its molecular weight (MW)

## 1.1: Biomass Reaction

The biomass reaction is formulated as the sum of its components, each with a stoichiometric coefficient representing the mass fraction *p* (relative proportion) of that component:

**(1)** $$\text{For } i \in n \text{, } p_{i=1}*\text{biomass}_{i=1} + p_{i=2}*\text{biomass}_{i=2} + ... + p_{i=n}*\text{biomass}_{i=n} \rightarrow \text{biomass}_\text{tot}$$
$$\sum_{i=1}^{n} p_{i} = 1$$

The Recon2.2 biomass function specifically looks as follows:

In [3]:
full_model.reactions.biomass_reaction.reaction

'0.014 biomass_DNA_c + 0.058 biomass_RNA_c + 0.071 biomass_carbohydrate_c + 0.097 biomass_lipid_c + 0.054 biomass_other_c + 0.706 biomass_protein_c --> biomass_c'

The mass fractions are:

In [4]:
mass_fractions = {metab.id: abs(coef) for metab, coef in \
                  full_model.reactions.biomass_reaction.metabolites.items() if coef < 1}
mass_fractions

{'biomass_protein_c': 0.706,
 'biomass_DNA_c': 0.014,
 'biomass_RNA_c': 0.058,
 'biomass_carbohydrate_c': 0.071,
 'biomass_lipid_c': 0.097,
 'biomass_other_c': 0.054}

The above mass fractions will be the default ones used by the ME Model, but may be changed by the user

We can see that the reactant coefficients sum to 1:

In [5]:
abs(sum([coef for coef in full_model.reactions.biomass_reaction.metabolites.values() if coef < 1]))

1.0

The biomass reaction has special units. 1g biomass<sub>tot</sub> must be produced per 1g DW<sub>cell</sub>, and 1g biomass<sub>tot</sub> = 1g DW<sub>cell</sub>

The basis is the product (i.e., coefficient units denominator is in 1g biomass<sub>tot</sub>)

Since the coefficients represent biomass fraction, each biomass component's coefficient is in $$\frac{\text{g biomass}_{i}}{\text{1g biomass}_\text{tot}} = \frac{\text{g biomass}_{i}}{\text{1g DW}_{\text{cell}}}$$<br> The coefficient proportions sum to 1 total, such that 1g biomass<sub>tot</sub> is produced per 1g DW<sub>cell</sub>

The biomass production rate converts to the growth rate as follows: $$\frac{\text{g biomass}_\text{tot}}{\text{g DW}_{\text{cell}}*\text{hr}} = \text{hr}^-1$$<br> 

## 1.2: Biomass Components

### 1.2.1 Explanation

Each biomass component is composed of its own metabolites with stoichiometric coefficients:

**(2)** $$\text{For } j \in m \text{, } s_{j=1,i}*\text{X}_{j=1,i} + s_{j=2,i}*\text{X}_{j=2,i} + ... + s_{j=m,i}*\text{X}_{j=m,i} \rightarrow \text{biomass}_{i}$$

For example, if we look at lipid biomass:

In [6]:
full_model.reactions.biomass_lipid.reaction

'0.210319587628866 chsterol_c + 0.120185567010309 clpn_hs_c + 0.240360824742268 pail_hs_c + 1.59237113402062 pchol_hs_c + 0.570865979381443 pe_hs_c + 0.0300412371134021 pglyc_hs_c + 0.0600927835051546 ps_hs_c + 0.180268041237113 sphmyln_hs_c --> biomass_lipid_c'

To understand units, it is important to note that the biomass  reaction can be combined with the reaction for formation of each component *i* into a single reaction. For each biomass component *i*, we can subsitute equation (2) into equation (1) to get: 

$$p_{1}*(s_{1,1}*\text{X}_{1,1} + s_{2,1}*\text{X}_{2,1} + ... + s_{m,1}*\text{X}_{m,1}) + p_{2}*(s_{1,2}*\text{X}_{1,2} + s_{2,2}*\text{X}_{2,2} + ... + s_{m,2}*\text{X}_{m,2}) + ... + p_{n}*(s_{1,n}*\text{X}_{1,n} + s_{2,n}*\text{X}_{2,n} + ... + s_{m,n}*\text{X}_{m,n}) \rightarrow \text{biomass}_\text{tot}$$

We refer to the above single reaction as the "standard formulation" as it is more commonly used to represent biomass objectives in metabolic models. We refer to the Recon2.2 biomass objective implemenation as the "component formation formulation". The component formation formulation is necessary for ME Models as it allows protein and RNA fractions to be variable.

From section 1.1, we know that p<sub>i</sub> is in units of $$\frac{\text{g biomass}_{i}}{\text{1g DW}_{\text{cell}}}$$

For consistency with the rest of the metabolic model (see section 1.0), we want p<sub>i</sub>s<sub>j</sub> to be in $$\frac{\text{mmol X}_{i,j}}{\text{1g DW}_{\text{cell}}}$$

Thus, s<sub>j</sub> should be in $$\frac{\text{mmol X}_{i,j}}{\text{g biomass}_{i}}$$

---
Again, the basis is the product (i.e., coefficient units denominator is in 1g biomass<sub>i</sub>)

To achieve mass balance and produce the stoichiometric total of the biomass component, given the units of s<sub>j</sub>, we expect that 

$$\sum_{j=1}^{m} s_{j,i}*\text{MW(X}_{j,i}) = 1$$

Again, this tells us that with the stoichiometric amounts of substrate, 1g biomass<sub>i</sub> is produced per 1g biomass<sub>i</sub> (unitless). This is then scaled in the biomass formation reaction by p<sub>i</sub> to produce p<sub>i</sub> g of biomass<sub>i</sub> per 1g biomass<sub>tot</sub>

---
In summary, looking again at equation (2):

Each substrate has coefficient s<sub>j,i</sub> units of $$\frac{\text{mmol X}_{j,i}}{\text{g biomass}_{i}}$$

Similar to equation (1), the production of component *i* in proportion p<sub>i</sub> to 1g biomass is equivalent to the growth rate:

$$\frac{\text{g biomass}_{i}}{\text{g biomass}_{i}*\text{hr}} = \text{hr}^-1$$

### 1.2.2 Corrections to Recon2.2 formulation

As mentioned, the stoichiometric coefficients of the reactions forming each biomass component must sum to 1 when scaling by the substrate's molecular weight. However, this is not the case for all the component formation reactions due to some errors. 

We are not concerned with biomass_protein and biomass_RNA, since these are formed separately in the ME model(see section 2)

Biomass carbohydrate is correctly formulated:

In [7]:
full_model.reactions.biomass_carbohydrate.reaction

'3.87591549295775 g6p_c --> biomass_carbohydrate_c'

The substrates sum to 1 g (sum output is in mg):

In [21]:
abs(sum([metab.formula_weight*coef for metab, coef in full_model.reactions.biomass_carbohydrate.metabolites.items()\
        if coef < 1])) 

1000.4509233266208

#### Biomass_DNA

In [9]:
full_model.reactions.biomass_DNA.reaction

'0.941642857142857 datp_n + 0.674428571428572 dctp_n + 0.707 dgtp_n + 0.935071428571429 dttp_n --> biomass_DNA_c'

We see that the stoichiometric coefficients of DNA biomass, scaled by their molecular weight, do not sum to 1g: 

In [10]:
abs(sum([metab.formula_weight*coef for metab, coef in full_model.reactions.biomass_DNA.metabolites.items()\
        if coef < 1])) 

1573.8843427801432

This is because the substrates are triphosphates, but when polymerized into DNA, the release of pyrophosphate should be considered. The number of pyrophosphates released = total number of NTP molecules

In [11]:
ppi_n = full_model.metabolites.ppi_n

n_released = abs(sum([coef for coef in full_model.reactions.biomass_DNA.metabolites.values() if coef<1]))

full_model.reactions.biomass_DNA.add_metabolites({ppi_n: n_released})

The new reaction is: 

In [12]:
full_model.reactions.biomass_DNA.reaction

'0.941642857142857 datp_n + 0.674428571428572 dctp_n + 0.707 dgtp_n + 0.935071428571429 dttp_n --> biomass_DNA_c + 3.2581428571428583 ppi_n'

Since we have a model metabolite as a product, we now expect the molecular weight scaled sum of coefficients of all substrates and products to be 1g:

In [13]:
abs(sum([metab.formula_weight*coef for metab, coef in full_model.reactions.biomass_DNA.metabolites.items()])) 

1003.8681381467144

#### Biomass_Lipid

In [22]:
full_model.reactions.biomass_lipid.reaction

'0.210319587628866 chsterol_c + 0.120185567010309 clpn_hs_c + 0.240360824742268 pail_hs_c + 1.59237113402062 pchol_hs_c + 0.570865979381443 pe_hs_c + 0.0300412371134021 pglyc_hs_c + 0.0600927835051546 ps_hs_c + 0.180268041237113 sphmyln_hs_c --> biomass_lipid_c'

Similarly, the lipid substrate stoichiometric coeffcieitns to not sum to 1g:

In [23]:
abs(sum([metab.formula_weight*coef for metab, coef in full_model.reactions.biomass_lipid.metabolites.items()\
        if coef < 1])) 

2195.3794614265366

Specifically, pchol_hs_c seems to be driving most of the deviation:

In [24]:
{metab: abs(metab.formula_weight*coef )for metab, coef in full_model.reactions.biomass_lipid.metabolites.items()\
        if coef < 1}

{<Metabolite chsterol_c at 0x7f4167895908>: 81.32081308804123,
 <Metabolite clpn_hs_c at 0x7f416789feb8>: 162.71273737645322,
 <Metabolite pail_hs_c at 0x7f41675e4b38>: 194.6980321341752,
 <Metabolite pchol_hs_c at 0x7f41675e7dd8>: 1168.8622935057742,
 <Metabolite pe_hs_c at 0x7f41675f00f0>: 395.0159213015874,
 <Metabolite pglyc_hs_c at 0x7f41675f5e48>: 21.688629255608276,
 <Metabolite sphmyln_hs_c at 0x7f416756cdd8>: 126.91520167676262,
 <Metabolite ps_hs_c at 0x7f41675a8d30>: 44.165833088133994}

Since we can't know the root driver of this imbalance, we simply scale the coefficients to sum to 1g:

In [25]:
norm_factor = abs(sum([metab.formula_weight*coef for metab, coef in \
                       full_model.reactions.biomass_lipid.metabolites.items() if coef < 1]))/1000 

new_reaction = {metab: coef/norm_factor for metab, coef in \
                full_model.reactions.biomass_lipid.metabolites.items() if coef < 1}
new_reaction[full_model.metabolites.biomass_lipid_c] = 1

full_model.reactions.biomass_lipid.add_metabolites(metabolites_to_add=new_reaction, 
                                                       combine = False)

In [26]:
abs(sum([metab.formula_weight*coef for metab, coef in full_model.reactions.biomass_lipid.metabolites.items()\
        if coef < 1])) 

1000.0

The new lipid reaction looks like this:

In [19]:
full_model.reactions.biomass_lipid.reaction

'0.09580101814936463 chsterol_c + 0.05474478062767954 clpn_hs_c + 0.10948486535720975 pail_hs_c + 0.7253284281824849 pchol_hs_c + 0.26003066413425396 pe_hs_c + 0.01368384720784515 pglyc_hs_c + 0.02737239031383982 ps_hs_c + 0.08211247504336976 sphmyln_hs_c --> biomass_lipid_c'

#### Biomass Other

biomass_other represents all components of biomass not included in carbohydrates, lipids, DNA, RNA, or protein. In Recon2.2, this represents 5.4% of the total biomass:

In [20]:
mass_fractions['biomass_other_c']

0.054

Yet, in Recon2.2, the biomass_other formation reaction is a demand reaction:

In [21]:
full_model.reactions.biomass_other.reaction

' --> biomass_other_c'

This means that 5.4% of the biomass is being represented by this pseudo-reaction. We can assume that this is absorption of vitamins, co-factors, and other nutrients from the extracellular environment that contribute to biomass come at no cost to metabolism. While this is the Recon2.2 implementation, it is more accurate to have a other component formation reaction with actual substrates here. 

In [45]:
# **ToDO**: remove this code
# This means that 5.4% of biomass can be created from nothing, which will cause an overestimation of growth rate. Instead, we distribute this 5.4% biomass across the other components and remove biomass_other altogether. There are 5 biomass components, excluding biomass_other. Thus, each components portion of biomass should increase by 0.054/5. These will be the default mass fractions used by the ME Model, though they can be modified by the user so long as they sum to 1:

# me_mass_fractions = {biomass_component: mass_frac + mass_fractions['biomass_other_c']/5 for \
#                      biomass_component, mass_frac in mass_fractions.items() if biomass_component != 'biomass_other_c'}
# me_mass_fractions

## 1.3: Biomass Consumption

Finally, there is a reaction to consume the generated biomass and prevent it from accumulating (generating mass balance). The reaction is as follows:

In [5]:
full_model.reactions.EX_biomass_c.reaction

'biomass_c <=> '

Technically, this reaction is for consumption of biomass and should not be reversible. But, because no other reaction consumes biomass, this reaction will never flow in the reverse direction since it would cause accumulation of biomass and violate the steady-state constraint of FBA (Sv=0).

# 2: ME Model Implementation

In addition to the above corrections, the biomass objective in the ME Model is modified since variable amounts of RNA and protein will be produced by the gene expression modules

## 2.1: Biomass Reaction

In the ME Model, each biomass component has a separate 1:1 input to total biomass.

Basically, 

(3) $$\text{for each } i \in n \text{, biomass}_{i} \rightarrow \text{biomass}_\text{tot}$$

Interpretation of 1:1 stoichiometry: Each gram of the biomass component *i* produced contributes to 1 g of total biomass (each is contributing to total biomass production). This allows RNA and protein to be input to biomass at variable proportions (not fixed to proportions of other components as in the Recon2.2 metabolic model biomass reaction).  

Using this logic, in terms of units, the biomass component coefficient is in $$\frac{\text{g biomass}_{i}}{\text{g biomass}_{i}}$$ whereas total biomass is in $$\frac{\text{g biomass}_\text{tot}}{\text{g biomass}_\text{tot}}$$ 

Here the coefficients become unitless, simply ensuring that total biomass depends on component *i*
with this 1:1 stoichiometric ratio.

Using the same logic as for equations (1) and (2), total biomass production simplifies to hr<sup>-1</sup>. All these reactions have open flux bounds [0,1000]; they are constrained by growth rate as delineated in the proceeding sections.

## 2.2 Biomass Components

### 2.2.1 Non-RNA/Protein Biomass 

Since the biomass reaction is now 1:1 for each component, the reactions forming the components must be scaled by the mass fractions of the original reaction to ensure proportionality

Thus equation (2) for a biomass component biomass<sub>i</sub> that had mass fraction p<sub>i</sub> from equation (1) will be formulated as follows for the ME Model:

**(4)** $$p_{i}s_{1,i}*\text{X}_{1,i} + p_{i}s_{2,i}*\text{X}_{2,i} + ... + p_{i}s_{m,i}*\text{X}_{m,i} \rightarrow p_{i}\text{biomass}_{i}$$

---
Here, units are the same as in the metabolic model. So, at stoichiometric ratios of p<sub>i</sub>s<sub>j,i</sub>, input metabolites in $$\frac{\text{mmol X}_{i,j}}{\text{1g DW}_{\text{cell}}}$$ produce p<sub>i</sub> counts of biomass<sub>i</sub> in $$\frac{\text{g biomass}_{i}}{\text{1g biomass}_\text{tot}}$$

The flux bounds of equation (4) are constrained to growth [mu, mu]. Since biomass<sub>tot</sub> directly determines growth rate (see section 2.3), by constraining the flux bounds of this reaction by growth, only p<sub>i</sub> counts of biomass<sub>i</sub> can contribute to biomass<sub>tot</sub> production in the 1:1 reaction from section 2.1. In other words, when flux = growth rate, the production rate of biomass<sub>i</sub> is p<sub>i</sub>*(growth rate) 

Note, using the same logic as before, we expect the coefficients scaled by molecular weight to sum to the mass fraction:

$$\sum_{j=1}^{m} p_{i}s_{j,i}*\text{MW(X}_{j,i}) = p_{i}$$

---

Taking the lipid component as an example, we see that p<sub>i</sub> in this case is 0.097:

In [46]:
mass_fractions['biomass_lipid_c']

0.097

The original stoichiometric coefficients looked like this: 

In [24]:
full_model.reactions.biomass_lipid.reaction

'0.09580101814936463 chsterol_c + 0.05474478062767954 clpn_hs_c + 0.10948486535720975 pail_hs_c + 0.7253284281824849 pchol_hs_c + 0.26003066413425396 pe_hs_c + 0.01368384720784515 pglyc_hs_c + 0.02737239031383982 ps_hs_c + 0.08211247504336976 sphmyln_hs_c --> biomass_lipid_c'

Thus, the ME Model reactions associated with lipid will be the following:

In [25]:
biomass_lipid = cobra.Reaction('lipid_biomass_to_biomass')
biomass_lipid.add_metabolites({full_model.metabolites.biomass_lipid_c: -1, 
                                            full_model.metabolites.biomass_c: 1})
biomass_lipid.reaction

'biomass_lipid_c --> biomass_c'

In [31]:
lipid_formation = cobra.Reaction('lipid_biomass_formation')
lipid_formation.add_metabolites({metab: coef*me_mass_fractions['biomass_lipid_c'] for metab, coef in \
 full_model.reactions.biomass_lipid.metabolites.items()})
lipid_formation.reaction

'0.010327349756501509 chsterol_c + 0.005901487351663855 clpn_hs_c + 0.01180246848550721 pail_hs_c + 0.07819040455807187 pchol_hs_c + 0.02803130559367258 pe_hs_c + 0.0014751187290057072 pglyc_hs_c + 0.0029507436758319325 ps_hs_c + 0.008851724809675261 sphmyln_hs_c --> 0.1078 biomass_lipid_c'

With this second reaction being bounder by growth rate

### 2.2.2 RNA/Protein Biomass 

With this formulation, RNA and protein biomass can be variable. Specifically, while the total mass fraction consisting of RNA and protein is still fixed to 76.4% (see cell below), the mass fraction of each RNA and DNA can vary within this total amount. The total fraction is fixed by the inclusion of p<sub>i</sub> from section 2.2.1, which requires proportional production of the other biomass components

In [47]:
mass_fractions['biomass_protein_c'] + mass_fractions['biomass_RNA_c']

0.764

Again, for each component, we have a 1:1 stoichiometry between total biomass and the component:

$$\text{biomass}_\text{RNA} \rightarrow \text{biomass}_\text{tot}$$
$$\text{biomass}_\text{protein} \rightarrow \text{biomass}_\text{tot}$$

But now, the amount of the component produced depends on the gene expression module. For example, if we look at a simplified generic protein synthesis reaction that produces *k* units of protein<sub>A</sub> (not all substrates/products are included for this toy example, all that matters is the amount of protein being produced):

$$\text{X amino_acids} \rightarrow k*\text{protein}_{A}$$

In the ME Model, we treat protein and RNA as metabolites. Remember that the standard metabolic model units for stoichiometric coefficients are $$\frac{\text{mmol X}}{\text{1g DW}_{\text{cell}}}$$ So, we can get the total grams of protein biomass being produced per g DW<sub>cell</sub> by this synthesis reaction by scaling the coefficient by the molecular weight of protein<sub>A</sub>. We can then include this directly in the synthesis reaction as follows:

$$\text{X aa} \rightarrow k*\text{protein}_{A} + k*\text{MW(protein)}_{A}*\text{biomass}_\text{protein}$$

The actual gene expression module is much more complex, and molecular weight gains or losses (e.g., in degradation reactions) in RNA and protein biomass are calculated directly in each reaction (see human_me.core.biomass.add_biomass_change for implementation).

### 2.2.3 Unmodeled and Orphan Protein Mass Fraction

**TODO**: add here

See [Google Docs](https://docs.google.com/document/d/1Hm3M7gtF-1ZdX_fLXJlb68KTrjDhXL4C_jLDXsHenqU/edit?usp=sharing) for details

### 2.2.4 Growth Associated Maintenance (GAM)

Many biomass objectives include an ATP hydrolysis that represents GAM energetic costs. In metabolic models, when the biomass objective is represented by the components formation formulation rather than the standard formulation, the GAM ATP hydrolysis term is typically included in the protein formation component, as can be seen for recon2.2: 

In [ ]:
## to do: show recon2.2 protein biomass formation reaction

This is likely because macromolecular synthesis costs are often used as a proxy for  GAM ([step 32](https://www.nature.com/articles/nprot.2009.203)). the ME model explicitly accounts for transcript and protein polymerization costs which constitute a large fraction of macromolecular synthesis costs ([source](https://doi.org/10.1073/pnas.1514974112)). While there are other energetic costs which are not accounted for in the ME Model, e.g. error-checking & replication ([review](https://doi.org/10.1016/j.mib.2010.03.003)), removal of the explicit GAM ATP hydrolysis term by the ME Model should not substantially underestimate the GAM costs. 

# 2.3 Biomass Objective

In the ME Model, we rcreate a "biomass_dilution" reaction that is both analogous to the biomass consumption reaction in section 1.3 and serves as the objective function for growth:

In [35]:
biomass_dilution = cobra.Reaction('biomass_dilution')
biomass_dilution.add_metabolites({full_model.metabolites.biomass_c: -1})
biomass_dilution.reaction

'biomass_c --> '

This reaction is also bounded by growth.